# Giai đoạn 2: Thu thập dữ liệu thời tiết 10 năm (2016-2026)

Quy trình thu thập dữ liệu khí tượng theo giờ (Hourly Weather Data) từ thư viện **Meteostat** cho 3 thành phố đại diện tại Việt Nam:
1. **Hà Nội (Trạm Nội Bài):** ID `48820`
2. **Đà Nẵng (Trạm Đà Nẵng):** ID `48855`
3. **TP. Hồ Chí Minh (Trạm Tân Sơn Nhất):** ID `48900`

Dữ liệu được chia nhỏ tải theo từng năm để tránh quá tải kết nối mạng và được lưu trực tiếp vào cơ sở dữ liệu SQLite cục bộ.

In [1]:
import sqlite3
from datetime import datetime
import pandas as pd
from meteostat import hourly

/Users/quangnguyentamhuu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Khởi tạo Cơ sở dữ liệu SQLite
Chúng ta thiết lập bảng `hourly_weather` để lưu trữ dữ liệu thời tiết thực tế theo giờ. Cặp khóa `(station_id, time)` được cấu hình làm Khóa chính (Primary Key) nhằm chống trùng lặp dữ liệu khi chạy lại tiến trình thu thập.

In [2]:
def init_db():
    conn = sqlite3.connect("../data/database/weather_data.db")
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS hourly_weather (
        city TEXT,
        station_id TEXT,
        time TEXT,
        temp REAL,
        dwpt REAL,
        rhum REAL,
        prcp REAL,
        snow REAL,
        wdir REAL,
        wspd REAL,
        wpgt REAL,
        pres REAL,
        tsun REAL,
        cldc REAL,
        coco INTEGER,
        PRIMARY KEY (station_id, time)
    )
    """)
    conn.commit()
    conn.close()
    print("Đã khởi tạo database '../data/database/weather_data.db' thành công.")

init_db()

Đã khởi tạo database '../data/database/weather_data.db' thành công.


## 2. Hàm tải dữ liệu phân đoạn và lưu vào Database

In [3]:
def download_data_for_city(city_name, station_id, start_year=2016, end_year=2026):
    conn = sqlite3.connect("../data/database/weather_data.db")
    
    for year in range(start_year, end_year + 1):
        start_date = datetime(year, 1, 1, 0, 0, 0)
        current_time = datetime.now()
        if year == current_time.year:
            end_date = current_time
        else:
            end_date = datetime(year, 12, 31, 23, 59, 59)
            
        print(f"Đang tải {city_name} năm {year} (từ {start_date.strftime('%Y-%m-%d')} đến {end_date.strftime('%Y-%m-%d')})...")
        
        try:
            # Gọi API Meteostat lấy dữ liệu theo giờ
            data = hourly(station_id, start_date, end_date)
            df = data.fetch()
            
            if df.empty:
                print(f"  [CẢNH BÁO] Không có dữ liệu cho năm {year}.")
                continue
                
            # Đưa cột index 'time' thành cột dữ liệu bình thường và định dạng chuỗi
            df = df.reset_index()
            df['time'] = df['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
            
            # Thêm cột thông tin thành phố và mã trạm
            df['city'] = city_name
            df['station_id'] = station_id
            
            # Đảm bảo thứ tự cột
            columns_order = [
                'city', 'station_id', 'time', 'temp', 'dwpt', 'rhum', 
                'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'cldc', 'coco'
            ]
            # Điền NaN cho cột còn thiếu
            for col in columns_order:
                if col not in df.columns:
                    df[col] = None
                    
            df_to_save = df[columns_order]
            
            # Lưu dữ liệu vào DB (chèn đè nếu trùng khóa chính)
            df_to_save.to_sql("hourly_weather", conn, if_exists="append", index=False)
            print(f"  [OK] Đã lưu {len(df_to_save)} dòng của năm {year} vào database.")
            
        except Exception as e:
            print(f"  [LỖI] Lỗi khi tải dữ liệu năm {year}: {str(e)}")
            
    conn.close()
    print(f"Hoàn thành thu thập dữ liệu cho {city_name}.")

## 3. Tiến hành tải dữ liệu cho 3 thành phố
Tiến hành gọi hàm tải cho Hà Nội, Đà Nẵng và TP. Hồ Chí Minh.

In [4]:
cities = {
    "Ha Noi": "48820",
    "Da Nang": "48855",
    "TP HCM": "48900"
}

for city, station_id in cities.items():
    download_data_for_city(city, station_id, start_year=2016, end_year=2026)

Đang tải Ha Noi năm 2016 (từ 2016-01-01 đến 2016-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2016: UNIQUE constraint failed: hourly_weather.station_id, hourly_weather.time
Đang tải Ha Noi năm 2017 (từ 2017-01-01 đến 2017-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2017: UNIQUE constraint failed: hourly_weather.station_id, hourly_weather.time
Đang tải Ha Noi năm 2018 (từ 2018-01-01 đến 2018-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2018: UNIQUE constraint failed: hourly_weather.station_id, hourly_weather.time
Đang tải Ha Noi năm 2019 (từ 2019-01-01 đến 2019-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2019: UNIQUE constraint failed: hourly_weather.station_id, hourly_weather.time
Đang tải Ha Noi năm 2020 (từ 2020-01-01 đến 2020-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2020: UNIQUE constraint failed: hourly_weather.station_id, hourly_weather.time
Đang tải Ha Noi năm 2021 (từ 2021-01-01 đến 2021-12-31)...
  [LỖI] Lỗi khi tải dữ liệu năm 2021: UNIQUE constraint failed: hourly_weather.station_id, 

## 4. Thống kê tổng số lượng dữ liệu thu được

In [5]:
conn = sqlite3.connect("../data/database/weather_data.db")
df_summary = pd.read_sql_query("""
    SELECT city, COUNT(*) as [Tổng số dòng], MIN(time) as [Từ ngày], MAX(time) as [Đến ngày] 
    FROM hourly_weather 
    GROUP BY city
""", conn)

print("=== THỐNG KÊ DỮ LIỆU ĐÃ THU THẬP ===")
display(df_summary)

total = pd.read_sql_query("SELECT COUNT(*) FROM hourly_weather", conn).iloc[0, 0]
print(f"\n=> TỔNG CỘNG: {total} dòng dữ liệu theo giờ đã được lưu trữ trong file '../data/database/weather_data.db'.")
conn.close()

=== THỐNG KÊ DỮ LIỆU ĐÃ THU THẬP ===


,city,Tổng số dòng,Từ ngày,Đến ngày
0,Da Nang,90943,2016-01-01 00:00:00,2026-05-22 22:00:00
1,Ha Noi,91036,2016-01-01 00:00:00,2026-05-22 22:00:00
2,TP HCM,91031,2016-01-01 00:00:00,2026-05-22 22:00:00



=> TỔNG CỘNG: 273010 dòng dữ liệu theo giờ đã được lưu trữ trong file '../data/database/weather_data.db'.
